In [4]:
import json
import datetime
from pathlib import Path
from typing import List, Dict, Optional
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import ThreadedPdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.settings import settings

# GPU Batch-Size für Seiten-Verarbeitung
settings.perf.page_batch_size = 64

In [5]:
class SimplePaperProcessor:
    def __init__(self, source_dir: str, test_mode: bool = True):
        self.source_dir = Path(source_dir)
        self.test_mode = test_mode

        # Timestamp für Output-Ordner
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_dir = self.source_dir.parent / "docling_output" / timestamp
        self.output_dir.mkdir(exist_ok=True, parents=True)

        # Test-Limit
        self.MAX_PDFS = 10 if test_mode else None

        # GPU Accelerator Setup
        accelerator_options = AcceleratorOptions(
            device=AcceleratorDevice.CUDA,
            num_threads=8
        )

        # Docling Setup
        pipeline_options = ThreadedPdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            accelerator_options=accelerator_options,
            layout_batch_size=64,
            table_batch_size=8,
        )
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )

        # Chunker
        self.chunker = HybridChunker(
            tokenizer="BAAI/bge-small-en-v1.5",
            max_tokens=300,
            overlap=20
        )

        print(f"📁 Output-Ordner: {self.output_dir.name}")
        if test_mode:
            print(f"🧪 Test-Mode: Erste {self.MAX_PDFS} PDFs")

    def build_section_index(self, doc) -> List[Dict]:
        """Baut Index aller Section-Headers mit Page/Position"""
        sections = []

        for item in doc.texts:
            if hasattr(item, 'label') and str(item.label) == 'section_header':
                if hasattr(item, 'prov') and item.prov:
                    sections.append({
                        "text": item.text.strip(),
                        "page": item.prov[0].page_no,
                        "top": item.prov[0].bbox.t
                    })

        return sections

    def extract_chunk_metadata(self, chunk) -> Dict:
        """Extrahiert Metadaten aus einem Chunk"""
        page_no = None
        section = None

        try:
            if chunk.meta and chunk.meta.doc_items:
                for doc_item in chunk.meta.doc_items:
                    if hasattr(doc_item, 'prov') and doc_item.prov:
                        page_no = doc_item.prov[0].page_no
                        break

            if chunk.meta and chunk.meta.headings:
                section = " > ".join(chunk.meta.headings)
        except Exception:
            pass

        return {"page": page_no, "section": section}

    def load_cached_chunks(self, output_file: Path) -> Optional[List[Dict]]:
        """Lädt gecachte Chunks"""
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data.get("chunks", None)
        except (json.JSONDecodeError, KeyError, IOError) as e:
            print(f"   ⚠️ Cache korrupt: {e}")
            return None

    def build_metainfo(self, pdf_name: str, doc, sections: List[Dict], all_chunks: List[Dict]) -> Dict:
        """Builds a metainfo block for the document"""
        num_pages = len(doc.pages) if hasattr(doc, 'pages') and doc.pages else 0
        num_tables = len(doc.tables) if hasattr(doc, 'tables') and doc.tables else 0
        num_pictures = len(doc.pictures) if hasattr(doc, 'pictures') and doc.pictures else 0
        num_equations = len(doc.equations) if hasattr(doc, 'equations') and doc.equations else 0

        num_text_blocks = 0
        num_footnotes = 0
        if hasattr(doc, 'texts') and doc.texts:
            num_text_blocks = len(doc.texts)
            num_footnotes = sum(
                1 for t in doc.texts
                if hasattr(t, 'label') and str(t.label) == 'footnote'
            )

        total_tokens = sum(c.get("token_count", 0) for c in all_chunks)

        return {
            "pdf_name": pdf_name,
            "num_pages": num_pages,
            "num_sections": len(sections),
            "num_text_blocks": num_text_blocks,
            "num_tables": num_tables,
            "num_pictures": num_pictures,
            "num_equations": num_equations,
            "num_footnotes": num_footnotes,
            "num_text_chunks": len(all_chunks),
            "total_tokens": total_tokens,
        }

    def process_converted_doc(self, pdf_path: Path, result) -> Optional[List[Dict]]:
        """Verarbeitet ein konvertiertes Dokument"""
        pdf_name = pdf_path.stem
        output_file = self.output_dir / f"{pdf_name}_chunks.json"

        try:
            print(f"📄 {pdf_name}")
            doc = result.document
            all_chunks = []

            # Section-Index bauen
            sections = self.build_section_index(doc)
            print(f"   📑 {len(sections)} Sections gefunden")

            # Nur Text-Chunks (Tabellen überspringen)
            text_idx = 0
            for chunk in self.chunker.chunk(doc):
                is_table = False
                if chunk.meta and chunk.meta.doc_items:
                    for doc_item in chunk.meta.doc_items:
                        if hasattr(doc_item, 'label') and str(doc_item.label) == 'table':
                            is_table = True
                            break
                if is_table:
                    continue

                meta = self.extract_chunk_metadata(chunk)
                all_chunks.append({
                    "chunk_id": f"{pdf_name}_text_{text_idx}",
                    "pdf_name": pdf_name,
                    "type": "text",
                    "page": meta["page"],
                    "section": meta["section"],
                    "token_count": self.chunker.tokenizer.count_tokens(chunk.text),
                    "text": chunk.text
                })
                text_idx += 1

            # Build metainfo
            metainfo = self.build_metainfo(pdf_name, doc, sections, all_chunks)

            # Speichern
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump({
                    "pdf_name": pdf_name,
                    "metainfo": metainfo,
                    "num_chunks": len(all_chunks),
                    "chunks": all_chunks
                }, f, indent=2, ensure_ascii=False)

            print(f"   ✓ {len(all_chunks)} Text-Chunks")

            return all_chunks

        except Exception as e:
            print(f"   ❌ Fehler: {str(e)[:50]}...")
            import traceback
            traceback.print_exc()
            return None

    def process_all(self) -> Dict:
        """Verarbeitet alle PDFs"""
        pdf_files = list(self.source_dir.glob("*.pdf"))

        if self.test_mode and self.MAX_PDFS:
            pdf_files = pdf_files[:self.MAX_PDFS]
            print(f"\n🧪 TEST MODE: {len(pdf_files)} PDFs\n")
        else:
            print(f"\n📚 Verarbeite {len(pdf_files)} PDFs\n")

        stats = {"success": 0, "cached": 0, "error": 0, "total_chunks": 0}

        to_convert = pdf_files

        if to_convert:
            print(f"\n🚀 Batch-Conversion: {len(to_convert)} PDFs...")
            results = self.converter.convert_all(
                [str(p) for p in to_convert],
                raises_on_error=False
            )

            for pdf_path, result in zip(to_convert, results):
                if result.status != ConversionStatus.SUCCESS:
                    print(f"⚠️ {pdf_path.stem} übersprungen (Status: {result.status})")
                    stats["error"] += 1
                    continue

                chunks = self.process_converted_doc(pdf_path, result)
                if chunks:
                    stats["success"] += 1
                    stats["total_chunks"] += len(chunks)
                else:
                    stats["error"] += 1

        print(f"\n{'='*40}")
        print(f"✅ Verarbeitet: {stats['success']}")
        print(f"❌ Fehler: {stats['error']}")
        print(f"📊 Total Chunks: {stats['total_chunks']}")

        return stats

In [6]:
processor = SimplePaperProcessor(
    source_dir="C:/Users/phili/PycharmProjects/UDA/data/src/fin_docs",
    test_mode=True
)
stats = processor.process_all()

2026-03-11 22:28:06,223 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-11 22:28:06,342 - INFO - Going to convert document batch...
2026-03-11 22:28:06,343 - INFO - Initializing pipeline for StandardPdfPipeline with options hash ced8fb7dc3d9965939dc2705856d65a3
2026-03-11 22:28:06,369 - INFO - Loading plugin 'docling_defaults'
2026-03-11 22:28:06,372 - INFO - Registered picture descriptions: ['vlm', 'api']


📁 Output-Ordner: 20260311_222805
🧪 Test-Mode: Erste 10 PDFs

🧪 TEST MODE: 10 PDFs


🚀 Batch-Conversion: 10 PDFs...


2026-03-11 22:28:06,398 - INFO - Loading plugin 'docling_defaults'
2026-03-11 22:28:06,405 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-03-11 22:28:06,431 - INFO - Loading plugin 'docling_defaults'
2026-03-11 22:28:06,436 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-03-11 22:28:06,482 - INFO - Accelerator device: 'cuda:0'
2026-03-11 22:28:07,117 - INFO - Loading plugin 'docling_defaults'
2026-03-11 22:28:07,119 - INFO - Registered table structure engines: ['docling_tableformer']
2026-03-11 22:28:07,299 - INFO - Accelerator device: 'cuda:0'
2026-03-11 22:28:07,784 - INFO - Processing document AAL_2010.pdf
2026-03-11 22:29:19,127 - INFO - Finished converting document AAL_2010.pdf in 72.91 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (595 > 512). Running this sequence through the model will result in indexi

📄 AAL_2010
   📑 445 Sections gefunden


2026-03-11 22:29:22,725 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-11 22:29:22,729 - INFO - Going to convert document batch...
2026-03-11 22:29:22,730 - INFO - Processing document AAL_2013.pdf


...

📄 AAPL_2004
   📑 345 Sections gefunden


2026-03-11 22:45:48,301 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-11 22:45:48,306 - INFO - Going to convert document batch...
2026-03-11 22:45:48,307 - INFO - Processing document AAPL_2005.pdf


   ✓ 410 Text-Chunks


2026-03-11 22:46:27,567 - INFO - Finished converting document AAPL_2005.pdf in 41.47 sec.


📄 AAPL_2005
   📑 338 Sections gefunden
   ✓ 411 Text-Chunks

✅ Verarbeitet: 10
❌ Fehler: 0
📊 Total Chunks: 7740
